# AquaSynex Phase 2.6: Official Machine Learning Benchmark (Google Colab)

**SIH26146 – AI-Powered Monitoring & Analysis of Bitcoin Transaction Traffic**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kushpagariya/AquaSynex-SIH26146/blob/ml/notebooks/08_official_colab_modeling.ipynb)

This notebook provides the **official, authoritative machine learning experimentation** for AquaSynex on the **hardened synthetic benchmark (`v2`)**.

> **Scientific Scope & Claim Boundary**:
> - This benchmark evaluates detection capabilities on the project-generated **hardened synthetic development dataset (`v2`)**.
> - Results demonstrate methodological validity on simulated UTXO topology and must **NOT** be construed as proving real-world Bitcoin criminal detection without empirical validation on live/historical network ledgers.

> **Strict Governance Protocols**:
> - **Test Quarantine**: The 15% out-of-time test partition ($N=1,500$) is **strictly quarantined, frozen, and untouched**.
> - **Zero Data Leakage**: All scalers and categorical encoders are fit **strictly on the training partition ($N=7,000$)**.
> - **Sequential Selection**: Model comparison is performed first; threshold tuning, multiclass evaluation, and SHAP explainability are conducted on the **provisional best binary model**.

In [1]:
# Cell 1: Environment Bootstrap & Dependency Setup
# When running in Google Colab, install required high-performance ML libraries
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('[*] Initializing Google Colab runtime environment...')
    # Package install for Colab:
    # !pip install -q duckdb catboost xgboost scikit-learn pandas numpy matplotlib seaborn pyyaml
    if not os.path.exists('data'):
        # !git clone -b ml https://github.com/kushpagariya/AquaSynex-SIH26146.git
        # %cd AquaSynex-SIH26146
        pass
else:
    print('[*] Running in local workstation environment.')

import os
import sys
import time
import json
import yaml
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except ImportError:
    sns = None

from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    log_loss,
    classification_report
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

print('[+] Libraries and dependencies successfully loaded.')


[*] Running in local workstation environment.
[+] Libraries and dependencies successfully loaded.


In [2]:
# Cell 2: Data Ingestion & Partition Verification
pq_path = 'data/processed/modeling_v2/modeling_dataset.parquet' if os.path.exists('data/processed/modeling_v2/modeling_dataset.parquet') else '../data/processed/modeling_v2/modeling_dataset.parquet'
manifest_path = 'data/processed/modeling_v2/feature_manifest.yaml' if os.path.exists('data/processed/modeling_v2/feature_manifest.yaml') else '../data/processed/modeling_v2/feature_manifest.yaml'

with open(manifest_path, 'r', encoding='utf-8') as f:
    manifest = yaml.safe_load(f)

canonical_features = manifest['canonical_feature_names']
categorical_cols = ['net_country', 'net_asn']
numeric_cols = [c for c in canonical_features if c not in categorical_cols]

con = duckdb.connect()
df = con.execute(f"SELECT * FROM read_parquet('{pq_path}')").df()
con.close()

df_train = df[df['temporal_split'] == 'train'].copy()
df_val = df[df['temporal_split'] == 'val'].copy()
df_test = df[df['temporal_split'] == 'test'].copy()

y_train = df_train['target_binary'].values
y_val = df_val['target_binary'].values
y_train_multi = df_train['target_multiclass'].values
y_val_multi = df_val['target_multiclass'].values

print('=== Partition Verification ===')
print(f'Total Records:     {len(df):,} rows x {len(df.columns)} columns')
print(f'Train Partition:   {len(df_train):,} rows (Suspicious: {y_train.mean():.2%})')
print(f'Val Partition:     {len(df_val):,} rows (Suspicious: {y_val.mean():.2%})')
print(f'Test Partition:    {len(df_test):,} rows [STRICTLY FROZEN & UNTOUCHED]')
print(f'Canonical Features: {len(canonical_features)} ({len(numeric_cols)} numeric + {len(categorical_cols)} categorical)')


=== Partition Verification ===
Total Records:     10,000 rows x 56 columns
Train Partition:   7,000 rows (Suspicious: 43.09%)
Val Partition:     1,500 rows (Suspicious: 42.27%)
Test Partition:    1,500 rows [STRICTLY FROZEN & UNTOUCHED]
Canonical Features: 46 (44 numeric + 2 categorical)


In [3]:
# Cell 3: Preprocessing Pipeline (Fit Strictly on Train)
scaler = RobustScaler()
X_train_num = scaler.fit_transform(df_train[numeric_cols])
X_val_num = scaler.transform(df_val[numeric_cols])

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_cat = ohe.fit_transform(df_train[categorical_cols])
X_val_cat = ohe.transform(df_val[categorical_cols])

cat_names = list(ohe.get_feature_names_out(categorical_cols))
all_feature_names = numeric_cols + cat_names

X_train = np.hstack([X_train_num, X_train_cat])
X_val = np.hstack([X_val_num, X_val_cat])

print(f'[+] Preprocessing complete: {X_train.shape[1]} total features ready for model training.')


[+] Preprocessing complete: 71 total features ready for model training.


In [4]:
# Cell 4: Candidate Binary Models Training & Validation
SEED = 42
models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_split=5,
        class_weight='balanced_subsample',
        random_state=SEED,
        n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=float(len(y_train) - sum(y_train)) / sum(y_train),
        eval_metric='logloss',
        random_state=SEED
    ),
    'CatBoost': CatBoostClassifier(
        iterations=400,
        depth=6,
        learning_rate=0.05,
        auto_class_weights='Balanced',
        eval_metric='Logloss',
        random_seed=SEED,
        verbose=0
    )
}

evaluations = {}
val_probs = {}

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - t0

    t_infer = time.time()
    probs = model.predict_proba(X_val)[:, 1]
    infer_lat = (time.time() - t_infer) / len(X_val) * 1000
    val_probs[name] = probs

    preds = (probs >= 0.50).astype(int)
    roc = roc_auc_score(y_val, probs)
    pr = average_precision_score(y_val, probs)
    acc = accuracy_score(y_val, preds)
    prec = precision_score(y_val, preds, zero_division=0)
    rec = recall_score(y_val, preds, zero_division=0)
    f1 = f1_score(y_val, preds, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_val, preds).ravel()

    evaluations[name] = {
        'ROC-AUC': round(roc, 4),
        'PR-AUC': round(pr, 4),
        'F1-Score': round(f1, 4),
        'Recall': round(rec, 4),
        'Precision': round(prec, 4),
        'Accuracy': round(acc, 4),
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
        'Fit Time (s)': round(fit_time, 2),
        'Latency (ms/1k)': round(infer_lat * 1000, 2)
    }

df_bench = pd.DataFrame(evaluations).T
print('=== Candidate Model Validation Benchmark (N=1,500) ===')
print(df_bench.to_string())


=== Candidate Model Validation Benchmark (N=1,500) ===
               ROC-AUC  PR-AUC  F1-Score  Recall  Precision  Accuracy     TN    FP    FN     TP  Fit Time (s)  Latency (ms/1k)
Random Forest   0.9950  0.9936    0.9597  0.9574     0.9620    0.9660  842.0  24.0  27.0  607.0          0.74            24.16
XGBoost         0.9981  0.9976    0.9733  0.9763     0.9702    0.9773  847.0  19.0  15.0  619.0          2.50             2.67
CatBoost        0.9978  0.9972    0.9685  0.9685     0.9685    0.9733  846.0  20.0  20.0  614.0          2.43             2.87


In [5]:
# Cell 5: Model Selection & Threshold Calibration
# Model selection based on highest validation ROC-AUC / PR-AUC
selected_name = max(evaluations.keys(), key=lambda m: (evaluations[m]['ROC-AUC'], evaluations[m]['PR-AUC']))
print(f'[+] Selected Best Binary Architecture: {selected_name}')
best_probs = val_probs[selected_name]

# Threshold Sweep: tau in [0.01, 0.99]
thresholds = np.arange(0.01, 1.00, 0.01)
f1_scores = []
rec_scores = []
prec_scores = []

best_f1 = -1.0
tau_f1 = 0.50
r95_candidates = []

for tau in thresholds:
    p = (best_probs >= tau).astype(int)
    r = recall_score(y_val, p, zero_division=0)
    prec = precision_score(y_val, p, zero_division=0)
    f = f1_score(y_val, p, zero_division=0)
    f1_scores.append(f)
    rec_scores.append(r)
    prec_scores.append(prec)

    if f > best_f1:
        best_f1 = f
        tau_f1 = tau
    if r >= 0.95:
        r95_candidates.append((tau, prec, f, r))

# Deterministic R95 threshold rule:
# Select threshold with highest Precision among Recall >= 0.95; tie-break highest F1
r95_candidates.sort(key=lambda item: (item[1], item[2]), reverse=True)
tau_r95 = r95_candidates[0][0]

def get_threshold_metrics(tau):
    p = (best_probs >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, p).ravel()
    return {
        'Threshold': round(float(tau), 2),
        'Precision': round(float(precision_score(y_val, p, zero_division=0)), 4),
        'Recall': round(float(recall_score(y_val, p, zero_division=0)), 4),
        'F1-Score': round(float(f1_score(y_val, p, zero_division=0)), 4),
        'Accuracy': round(float(accuracy_score(y_val, p)), 4),
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)
    }

thresh_table = pd.DataFrame([
    {'Operating Point': 'Default (tau=0.50)', **get_threshold_metrics(0.50)},
    {'Operating Point': f'F1-Optimal (tau={tau_f1:.2f})', **get_threshold_metrics(tau_f1)},
    {'Operating Point': f'High-Recall R95 (tau={tau_r95:.2f})', **get_threshold_metrics(tau_r95)}
])

print('=== Decision Threshold Operating Points (Validation Partition) ===')
print(thresh_table.to_string(index=False))


[+] Selected Best Binary Architecture: XGBoost
=== Decision Threshold Operating Points (Validation Partition) ===
           Operating Point  Threshold  Precision  Recall  F1-Score  Accuracy  TN  FP  FN  TP
        Default (tau=0.50)       0.50     0.9702  0.9763    0.9733    0.9773 847  19  15 619
     F1-Optimal (tau=0.32)       0.32     0.9632  0.9921    0.9775    0.9807 842  24   5 629
High-Recall R95 (tau=0.67)       0.67     0.9821  0.9543    0.9680    0.9733 855  11  29 605


In [6]:
# Cell 6: Secondary 11-Class Multiclass Experiment
print('=== Secondary Experiment: 11-Class Scenario Attribution ===')
classes = sorted(list(np.unique(y_train_multi)))
class_to_idx = {c: i for i, c in enumerate(classes)}
y_train_idx = np.array([class_to_idx[c] for c in y_train_multi])
y_val_idx = np.array([class_to_idx[c] for c in y_val_multi])

multi_cat = CatBoostClassifier(
    iterations=350,
    depth=6,
    learning_rate=0.06,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    random_seed=SEED,
    verbose=0
)
multi_cat.fit(X_train, y_train_idx)
val_multi_probs = multi_cat.predict_proba(X_val)
val_multi_preds = np.argmax(val_multi_probs, axis=1)

multi_acc = accuracy_score(y_val_idx, val_multi_preds)
multi_macro = f1_score(y_val_idx, val_multi_preds, average='macro', zero_division=0)
multi_weighted = f1_score(y_val_idx, val_multi_preds, average='weighted', zero_division=0)
multi_ll = log_loss(y_val_idx, val_multi_probs)

print(f'Multiclass Overall: Top-1 Acc = {multi_acc:.4f} | Macro F1 = {multi_macro:.4f} | Log-Loss = {multi_ll:.4f}')
rep = classification_report(y_val_idx, val_multi_preds, target_names=classes, digits=4, zero_division=0)
print()
print('Per-Scenario Performance Breakdown:')
print(rep)


=== Secondary Experiment: 11-Class Scenario Attribution ===
Multiclass Overall: Top-1 Acc = 0.9660 | Macro F1 = 0.9566 | Log-Loss = 0.1002

Per-Scenario Performance Breakdown:
                      precision    recall  f1-score   support

      amount_anomaly     1.0000    1.0000    1.0000         7
  benign_high_volume     1.0000    1.0000    1.0000        95
coordinated_activity     0.9730    0.9114    0.9412        79
         high_fan_in     1.0000    1.0000    1.0000        34
        high_fan_out     0.9412    1.0000    0.9697        32
         mixing_like     1.0000    0.9394    0.9688        33
              normal     0.9753    0.9741    0.9747       771
       peeling_chain     0.9847    0.9923    0.9885       130
      rapid_multihop     1.0000    1.0000    1.0000        94
    temporal_anomaly     0.8000    0.7500    0.7742        16
   transaction_burst     0.8930    0.9187    0.9057       209

            accuracy                         0.9660      1500
           macro

In [7]:
# Cell 7: Explainability & SHAP Analysis (Selected Binary Model Only)
selected_binary_model = models[selected_name]
print(f'=== Feature Attribution for Selected Binary Model: {selected_name} ===')

# Tree Gain Feature Importances
if hasattr(selected_binary_model, 'feature_importances_'):
    imp = selected_binary_model.feature_importances_
    top_idx = np.argsort(imp)[::-1][:20]
    df_imp = pd.DataFrame({
        'Feature': [all_feature_names[i] for i in top_idx],
        'Importance': [round(float(imp[i]), 5) for i in top_idx]
    })
    print('Top 20 Tree Feature Importances:')
    print(df_imp.to_string(index=False))

# SHAP Explainability (Validation Sample N=300)
try:
    import shap
    print()
    print('[*] Computing Tree SHAP values on validation sample...')
    sample_idx = np.random.default_rng(SEED).choice(len(X_val), size=300, replace=False)
    X_sample = X_val[sample_idx]
    explainer = shap.TreeExplainer(selected_binary_model)
    shap_vals = explainer.shap_values(X_sample)
    if isinstance(shap_vals, list) and len(shap_vals) == 2:
        shap_mat = shap_vals[1]
    else:
        shap_mat = shap_vals
    mean_shap = np.mean(np.abs(shap_mat), axis=0)
    top_shap_idx = np.argsort(mean_shap)[::-1][:15]
    df_shap = pd.DataFrame({
        'Feature': [all_feature_names[i] for i in top_shap_idx],
        'Mean |SHAP|': [round(float(mean_shap[i]), 5) for i in top_shap_idx]
    })
    print('Top 15 SHAP Attribution Drivers:')
    print(df_shap.to_string(index=False))
except Exception as e:
    print(f'[*] Python shap package fallback to native Tree SHAP: {e}')
    if hasattr(selected_binary_model, 'get_booster'):
        import xgboost as xgb
        dval = xgb.DMatrix(X_val, feature_names=all_feature_names)
        contribs = selected_binary_model.get_booster().predict(dval, pred_contribs=True)
        shap_mat = contribs[:, :-1]
        mean_shap = np.mean(np.abs(shap_mat), axis=0)
        top_shap_idx = np.argsort(mean_shap)[::-1][:15]
        df_shap = pd.DataFrame({
            'Feature': [all_feature_names[i] for i in top_shap_idx],
            'Mean |SHAP|': [round(float(mean_shap[i]), 5) for i in top_shap_idx]
        })
        print('Top 15 Native Tree SHAP Attribution Drivers (N=1,500 validation partition):')
        print(df_shap.to_string(index=False))


=== Feature Attribution for Selected Binary Model: XGBoost ===
Top 20 Tree Feature Importances:
                      Feature  Importance
hist_out_mean_neighbor_degree     0.13492
                tx_size_bytes     0.12823
               tx_input_count     0.07881
     hist_address_reuse_ratio     0.05893
time_since_prev_global_tx_sec     0.05430
             time_day_of_week     0.05107
             time_txs_last_1m     0.03328
              tx_output_count     0.02130
          hist_component_size     0.02003
       rel_change_value_ratio     0.01833
        tx_input_output_ratio     0.01640
                   rel_fan_in     0.01564
                   tx_log_fee     0.01517
                  rel_fan_out     0.01390
 hist_in_mean_neighbor_degree     0.01371
                  tx_fee_sats     0.01333
                net_asn_60068     0.01279
            hist_cluster_size     0.01263
                 net_asn_9009     0.01213
        hist_cluster_tx_count     0.01202
[*] Python shap packag

## 8. Official Phase 2.6 Conclusions & Recommendations

1. **Winning Architecture**: **XGBoost** achieved the highest overall validation discrimination with **0.9981 ROC-AUC**, **0.9976 PR-AUC**, and **0.9733 F1-Score** (at default $\tau = 0.50$), closely matched by **CatBoost** (**0.9978 ROC-AUC**, **0.9972 PR-AUC**).
2. **Threshold Optimization**: Calibrating the threshold to $\tau^*_{F_1} = 0.32$ improves validation Recall from **97.63% to 99.21%** while maintaining **96.32% Precision** ($F_1 = 0.9775$). For high-assurance monitoring, the deterministic $\tau^*_{R95} = 0.67$ provides **95.43% Recall with 98.21% Precision**.
3. **Secondary Multiclass Capability**: CatBoost demonstrates strong fine-grained attribution across all 11 scenarios with **96.60% Top-1 Accuracy** and **0.9566 Macro F1**.
4. **Authentic Feature Drivers**: As confirmed by feature importance and SHAP analysis, models now synthesize multi-input consolidation structure, historical graph neighbor degrees, and temporal inter-arrival cadence rather than artificial single-feature generator shortcuts.
5. **Test Set Status**: The 15% out-of-time test partition ($N=1,500$) remained **100% untouched and unblinded** throughout Phase 2.6.